# Modul 06 · Notebook 01 — GPU, Precision & Optimasi Inferensi

> **Domain sertifikasi NCA-GENL: Software Development (24%).**
> Notebook ini menyentuh inti "kenapa NVIDIA" di balik LLM modern: *kenapa GPU jauh lebih cepat untuk perkalian matriks*, bagaimana **precision** (FP32 / FP16 / FP8) menukar memori dan kecepatan, bagaimana **quantization 4-bit** memuat model besar di GPU kecil, lalu bagaimana **TensorRT** dan **TensorRT-LLM** meng-*compile* model jadi *engine* yang dioptimalkan untuk produksi. Semua angka di notebook ini dihasilkan dari kode yang benar-benar jalan di Colab T4 — bukan klaim kosong.

Inilah peta perjalanan Track 1 (NVIDIA Stack):

| nb | Fokus | Yang kamu bawa pulang |
|----|-------|-----------------------|
| **01 (ini)** | GPU, precision, quantization, TensorRT-LLM | Kenapa GPU cepat & cara memperkecil/mempercepat model |
| 02 | Serving: Triton -> Dynamo -> NIM | Cara menyajikan model ke banyak user |
| 03-05 | Trustworthy AI (fairness, guardrails, capstone) | Cara membuat layanan AI yang aman & adil |

## Kenapa ini penting?

Sebuah model bisa "pintar" tapi tetap **tidak bisa di-deploy** kalau ia tidak muat di GPU, atau terlalu lambat menjawab. Optimasi inferensi adalah jembatan antara *"model bagus di notebook riset"* dan *"layanan yang dipakai ribuan orang"*. Tiga tuas utama yang akan kita tarik:

1. **Precision** -- pakai angka yang lebih pendek (FP16/FP8 alih-alih FP32) -> memori turun, Tensor Core menyala.
2. **Quantization** -- kompres bobot ke 4-bit -> model 7B muat di GPU 16 GB.
3. **Compile** (TensorRT / TensorRT-LLM) -- ubah model jadi *engine* native GPU -> latency & throughput jauh lebih baik di produksi.

> WARNING: **Jalankan di Google Colab dengan GPU T4.** `Runtime -> Change runtime type -> T4 GPU`. Tanpa GPU, sel-sel benchmark akan gagal.

## 0. Persiapan -- dependensi + cek GPU

Kita pin `transformers<5` (4.53+ mengenali arsitektur Qwen3/Qwen2.5 dengan benar) dan menyiapkan `bitsandbytes` untuk quantization 4-bit nanti. `openai` dipakai di bagian akhir untuk menunjukkan kontrak NIM (lihat nb02 untuk detail penuh).

In [ ]:
# Dependensi (pinned untuk reproducibility di Colab T4)
!pip -q install "transformers<5" "accelerate>=0.30" "bitsandbytes>=0.43" openai
# Cek kartu grafis yang kita dapat
!nvidia-smi -L

Sebelum mengukur apa pun, pastikan PyTorch benar-benar melihat GPU. `compute capability` penting: Tensor Core untuk FP16 aktif sejak Volta (7.0); T4 adalah Turing (7.5). FP8 baru ada di Hopper (9.0) / Ada (8.9) ke atas -- jadi di T4 kita **menjelaskan** FP8 secara konsep, bukan menjalankannya.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU tidak aktif -- Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
print("GPU            :", torch.cuda.get_device_name(0))
print("Compute cap.   :", f"{cc[0]}.{cc[1]}", "(Turing/T4 -> Tensor Core FP16 ada, FP8 belum)")
print("Memori total   :", round(p.total_memory / 1e9, 1), "GB")
print("Memori dipakai :", round(torch.cuda.memory_allocated(0) / 1e9, 2), "GB")

## 1. Kenapa GPU untuk LLM? -- Tensor Core & paralelisme

Inti dari setiap layer transformer adalah **perkalian matriks** (matmul). Saat model menjawab, miliaran perkalian-dan-penjumlahan terjadi. Di sinilah CPU vs GPU berbeda secara fundamental:

| | CPU | GPU |
|---|-----|-----|
| Jumlah core | belasan, masing-masing "pintar" | ribuan, masing-masing sederhana |
| Cocok untuk | logika bercabang, tugas berurutan | operasi **sama** pada **banyak data** sekaligus |
| Analoginya | beberapa koki ahli | ribuan tangan mengaduk satu adonan raksasa |

Perkalian matriks adalah *embarrassingly parallel*: setiap elemen hasil dihitung independen, jadi ribuan core GPU bisa mengerjakannya **serentak**. Itu sebabnya matmul di GPU bisa puluhan kali lebih cepat daripada CPU.

### Tensor Core -- akselerator khusus matmul

Sejak arsitektur Volta, NVIDIA menanam **Tensor Core**: unit perangkat keras yang dalam satu operasi mengerjakan sebuah perkalian-akumulasi matriks kecil (mis. 4x4) langsung di silikon. Syaratnya: data dalam **precision rendah** (FP16/BF16/FP8/INT8), bukan FP32. Inilah kunci pertama kita:

> **FP16 bukan cuma menghemat memori -- ia menyalakan Tensor Core**, sehingga matmul jadi berkali-kali lebih cepat dibanding FP32 yang berjalan di CUDA core biasa.

Mari buktikan dengan benchmark nyata.

### Benchmark precision #1 -- kecepatan matmul (FP32 vs FP16)

Kita kalikan dua matriks 4096x4096 sebanyak 50 kali, sekali dalam FP32 dan sekali dalam FP16, lalu ukur waktu per matmul. Dua detail teknis yang **wajib** benar agar angkanya jujur:

- **Warmup** -- beberapa iterasi awal dibuang (alokasi & kompilasi kernel pertama selalu lambat).
- **`torch.cuda.synchronize()`** -- GPU bekerja *asynchronous*; tanpa sinkronisasi, `time.time()` berhenti sebelum GPU selesai dan angkanya palsu.

In [ ]:
import time

def matmul_bench(dtype, n=4096, iters=50):
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    b = torch.randn(n, n, device="cuda", dtype=dtype)
    for _ in range(5):                # warmup: buang kernel-launch pertama yang lambat
        _ = a @ b
    torch.cuda.synchronize()          # GPU async -> WAJIB sinkron sebelum start timer
    t0 = time.time()
    for _ in range(iters):
        _ = a @ b
    torch.cuda.synchronize()          # tunggu semua matmul benar-benar selesai
    return (time.time() - t0) / iters * 1000   # milidetik per matmul

ms32 = matmul_bench(torch.float32)
ms16 = matmul_bench(torch.float16)
print(f"Matmul 4096x4096 FP32 : {ms32:6.2f} ms  (CUDA core)")
print(f"Matmul 4096x4096 FP16 : {ms16:6.2f} ms  (Tensor Core menyala)")
print(f"Speedup FP16          : {ms32/ms16:.1f}x lebih cepat")

**Angka rujukan di T4** (akan mirip di mesinmu):

```
Matmul 4096x4096 FP32 : 34.81 ms
Matmul 4096x4096 FP16 :  6.23 ms
Speedup FP16          :  5.6x lebih cepat
```

Lima kali lebih cepat **hanya** dengan mengganti tipe angka -- itulah Tensor Core bekerja. Pada beban LLM nyata (banyak matmul beruntun), efek ini menumpuk menjadi selisih latency yang sangat terasa.

## 2. Tangga precision: FP32 -> FP16 -> FP8

Sebuah angka pecahan disimpan dalam beberapa "bit". Makin sedikit bit, makin kecil memori dan makin cepat hitungannya -- dengan ongkos ketelitian:

| Precision | Bit | Memori relatif | Tensor Core | Catatan |
|-----------|-----|----------------|-------------|---------|
| **FP32** | 32 | 1.0x (penuh) | tidak | presisi penuh, baseline riset |
| **FP16** | 16 | **0.5x** | ya (Volta+) | default deploy; kualitas ~ FP32 |
| **BF16** | 16 | 0.5x | ya (Ampere+) | rentang lebih lebar, **tidak optimal di T4** |
| **FP8** | 8 | **0.25x** | ya (Hopper/Ada+) | inference datacenter; butuh GPU baru |

Catatan praktis untuk T4: kita selalu pakai **FP16, bukan BF16**. BF16 baru optimal di Ampere (A100) ke atas dan bisa bermasalah saat dipasangkan dengan 4-bit di T4. **FP8** adalah arah masa depan (dipakai TensorRT-LLM di H100), tapi tidak bisa dijalankan di T4 -- jadi kita pahami konsepnya saja.

### Benchmark precision #2 -- memori bobot model (FP32 vs FP16)

Kalau matmul tadi mengukur *kecepatan*, sekarang kita ukur *memori*. Kita muat model yang sama (`Qwen2.5-1.5B-Instruct`) dua kali -- sekali FP32, sekali FP16 -- dan baca `torch.cuda.memory_allocated()` setelah bobot masuk GPU.

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)

def load_weight_mem(dtype):
    torch.cuda.empty_cache(); gc.collect()
    # transformers<5 menerima torch_dtype= di seluruh rentang 4.x (alias dtype= baru ada di 4.56+).
    model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map="auto")
    mem_gb = torch.cuda.memory_allocated() / 1e9     # memori yang ditempati bobot
    del model; gc.collect(); torch.cuda.empty_cache()
    return mem_gb

mem32 = load_weight_mem(torch.float32)
mem16 = load_weight_mem(torch.float16)
print(f"Memori bobot FP32 : {mem32:.2f} GB")
print(f"Memori bobot FP16 : {mem16:.2f} GB")
print(f"Penghematan       : {mem32/mem16:.1f}x lebih hemat")

**Angka rujukan di T4:**

```
Memori bobot FP32 : 6.18 GB
Memori bobot FP16 : 3.10 GB
Penghematan       : 2.0x lebih hemat
```

Persis sesuai teori: 16 bit = separuh dari 32 bit -> memori turun **2.0x**, dan (dari benchmark sebelumnya) matmul jadi **5.6x** lebih cepat. **FP16 adalah default deploy yang paling aman**: kualitas jawaban hampir identik dengan FP32, tapi separuh memori dan jauh lebih cepat. Dari sini, satu-satunya cara turun lebih jauh adalah **quantization**.

## 3. Quantization 4-bit untuk inference -- memuat yang tak muat

FP16 sudah memangkas memori 2x. **Quantization** mendorong lebih jauh: simpan bobot dalam **4-bit**. Analoginya seperti membulatkan harga ke ribuan terdekat -- ukuran datanya jauh lebih kecil, ketelitiannya sedikit berkurang, tapi makna utamanya bertahan.

Kita pakai dua trik dari `bitsandbytes`:

- **NF4** (`bnb_4bit_quant_type="nf4"`, *4-bit NormalFloat*) -- format 4-bit yang dirancang khusus mengikuti sebaran bobot neural network, jadi lebih akurat daripada pembulatan 4-bit biasa.
- **compute_dtype = FP16** (`bnb_4bit_compute_dtype=torch.float16`) -- bobot **disimpan** 4-bit, tapi saat dihitung di-*dequantize* ke FP16. Di T4 kita pakai FP16 (bukan BF16).
- **double quant** (`bnb_4bit_use_double_quant=True`) -- meng-quantize konstanta quantization itu sendiri; hemat sedikit memori lagi tanpa kehilangan akurasi berarti.

### Ini *inference* quant -- beda dengan QLoRA *training* di Modul 04

Di **Modul 04** kamu memakai 4-bit untuk **training** (QLoRA): model di-quantize agar muat, lalu *adapter* LoRA dilatih di atasnya. Tujuannya **mengubah** perilaku model.

Di sini tujuannya berbeda: model **sudah jadi**, kita hanya ingin **menjalankannya** dengan memori lebih kecil. Tidak ada training, tidak ada adapter -- murni kompresi untuk **deploy**. Konfigurasi `BitsAndBytesConfig`-nya kebetulan mirip, tapi niat dan konteksnya berlainan: *training* (M04) vs *inference* (M06).

In [ ]:
from transformers import BitsAndBytesConfig

# 1) Pembanding: muat FP16 lalu catat memorinya (lalu bebaskan)
torch.cuda.empty_cache(); gc.collect()
m_fp16 = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto")
mem_fp16 = torch.cuda.memory_allocated() / 1e9
del m_fp16; gc.collect(); torch.cuda.empty_cache()

# 2) Muat 4-bit (NF4, compute fp16, double-quant). Model ini KITA SIMPAN untuk generate.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16 (BUKAN bf16) -> aman di T4
    bnb_4bit_use_double_quant=True,
)
m_4bit = AutoModelForCausalLM.from_pretrained(NAME, quantization_config=bnb, device_map="auto")
mem_4bit = torch.cuda.memory_allocated() / 1e9

print(f"Memori bobot FP16  : {mem_fp16:.2f} GB")
print(f"Memori bobot 4-bit : {mem_4bit:.2f} GB")
print(f"Penghematan        : {mem_fp16/mem_4bit:.1f}x lebih hemat dari FP16")

**Angka rujukan di T4:**

```
Memori bobot FP16  : 3.09 GB
Memori bobot 4-bit : 1.16 GB
Penghematan        : 2.7x lebih hemat dari FP16
```

Dari **3.09 -> 1.16 GB**. Gabungkan dengan langkah FP32->FP16 sebelumnya, dan kita sudah memangkas dari ~6.2 GB (FP32) menjadi ~1.2 GB (4-bit) -- sekitar **5x**. Inilah yang membuat model 7B (yang butuh ~14 GB di FP16) bisa muat di T4 16 GB saat di-quantize.

### Tapi -- apakah jawabannya masih masuk akal?

Kompresi tidak gratis. Mari pastikan model 4-bit tetap menjawab dengan benar (bukan sekadar muat tapi ngawur).

In [ ]:
# Uji kualitas model 4-bit: dia masih bisa menjawab koheren?
ids = tok.apply_chat_template(
    [{"role": "user", "content": "Sebutkan 3 manfaat quantization saat menjalankan LLM. Jawab singkat."}],
    add_generation_prompt=True, return_tensors="pt").to(m_4bit.device)
out = m_4bit.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=140,
                      do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())

# Bersihkan memori sebelum lanjut
del m_4bit; gc.collect(); torch.cuda.empty_cache()

Jawaban tetap koheren dan relevan -- inilah inti quantization 4-bit untuk inference: **penurunan kualitas kecil, penghematan memori besar.**

### Kapan pakai apa?

| Situasi | Pilihan | Alasan |
|---------|---------|--------|
| Model muat nyaman di GPU | **FP16** | kualitas penuh, default aman |
| Model **tidak muat** (mis. 7B di T4) | **4-bit** | satu-satunya cara memuatnya |
| Ingin layani lebih banyak request paralel | 4-bit | sisa VRAM dipakai untuk batch lebih besar |
| Butuh kualitas riset/eval presisi | FP16/FP32 | hindari kehilangan akurasi quant |

## 4. Jalur kedua: *compile* model -- TensorRT & TensorRT-LLM

Sejauh ini kita **memperkecil** bobot (precision & quantization). Tuas ketiga berbeda: **meng-*compile*** model menjadi *engine* native yang dijahit khusus untuk satu GPU. Di sinilah ekosistem NVIDIA punya dua alat yang sering tertukar.

### TensorRT -- *compiler* inference umum

**TensorRT** adalah optimizer & runtime inference NVIDIA untuk model *apa pun* (CNN, ViT, deteksi objek, dll). Alurnya: `model -> ONNX -> TensorRT engine`. Yang ia lakukan:

- **Layer/kernel fusion** -- menggabungkan beberapa operasi (mis. conv + bias + ReLU) jadi satu kernel -> lebih sedikit lalu-lintas memori.
- **Precision calibration** -- memilih FP16/INT8 per-layer secara otomatis.
- **Kernel auto-tuning** -- memilih implementasi kernel tercepat untuk GPU target spesifik.

Hasilnya *engine* yang terikat pada GPU itu (engine yang di-build untuk T4 belum tentu jalan di A100). Build-nya cukup berat dan rapuh di Colab, jadi di kursus ini TensorRT kita **kenalkan sebagai konsep**.

> WARNING: **Resep terbukti (untuk eksplorasi mandiri):** pin `tensorrt<11`, `tf2onnx>=1.17`, dan ingat jalur **TF-TRT sudah mati di TF 2.18+** -> gunakan **ONNX -> TRT**. Jangan build engine penuh di Colab T4.

### TensorRT-LLM -- *compiler* khusus LLM

LLM punya kebutuhan unik yang tidak ditangani TensorRT umum: teks dihasilkan **token demi token** secara autoregresif, panjang prompt bervariasi, dan banyak request datang bersamaan. **TensorRT-LLM** adalah cabang khusus untuk ini. Fitur kuncinya:

| Fitur | Apa yang dipecahkan |
|-------|---------------------|
| **Paged KV-cache** | "Memori perhatian" model dipecah jadi blok-blok kecil (mirip paging OS) -> tak ada VRAM terbuang, muat lebih banyak request |
| **In-flight (continuous) batching** | Request baru bergabung ke batch **tanpa menunggu** request lama selesai -> GPU tak pernah menganggur |
| **FP8 / INT4 quantization** | Precision sangat rendah khusus inference LLM di GPU baru (Hopper/Ada) -> throughput maksimal |
| **Fused attention kernels** | Kernel khusus untuk attention -> latency per-token turun |

Singkatnya: **TensorRT = optimizer umum; TensorRT-LLM = optimizer LLM** dengan KV-cache & batching yang sadar-LLM. Untuk model bahasa, TensorRT-LLM-lah yang relevan.

> **Build engine TensorRT-LLM penuh hanya jalan di GPU NVIDIA x86 datacenter** (bukan Colab T4 free, bukan ARM/Jetson untuk build). Jadi kita tunjukkan **one-liner servernya sebagai referensi**, tidak dijalankan di sini.

### `trtllm-serve` -- server OpenAI dari engine TensorRT-LLM (referensi)

Setelah engine ter-build di host x86 NVIDIA, `trtllm-serve` mengangkatnya menjadi **server yang berbicara protokol OpenAI** `/v1` -- kontrak yang sama persis dengan yang kamu pakai di Modul 04 (Ollama/vLLM/NIM). Sel di bawah ini sengaja **tidak dieksekusi** (`%%script echo`) -- ia hanya mencetak perintah sebagai dokumentasi, supaya notebook tetap aman dijalankan di Colab.

In [ ]:
%%script echo --- REFERENSI (tidak dijalankan di Colab; butuh GPU x86 NVIDIA) ---
# 1) Install di host x86 NVIDIA (datacenter / workstation RTX, driver R560+):
pip install tensorrt-llm

# 2) Angkat model jadi server OpenAI-compatible.
#    trtllm-serve mem-build engine (paged KV-cache, in-flight batching, FP8/INT4) lalu serve:
trtllm-serve "Qwen/Qwen2.5-1.5B-Instruct" --host 0.0.0.0 --port 8000

# 3) Sekarang server bicara protokol OpenAI /v1 -- kode klien IDENTIK dengan NIM/Ollama:
#    from openai import OpenAI
#    client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
#    client.chat.completions.create(model="Qwen/Qwen2.5-1.5B-Instruct",
#                                   messages=[{"role":"user","content":"Halo!"}])

**Pesan inti -- "one client, many backends".** Perhatikan baris klien di komentar: ia **identik** dengan yang kamu tulis untuk Ollama di Modul 04, dan identik dengan kode NIM di nb02 berikutnya. Hanya `base_url` (dan `model`/`api_key`) yang berubah; logika aplikasimu tetap sama, apa pun backend-nya:

```
hosted NIM (cloud)  =  self-host NIM (RTX)  =  TensorRT-LLM engine  =  Ollama edge
        \________________ semuanya bicara OpenAI /v1 ________________/
```

Optimasi (precision, quant, TensorRT-LLM) terjadi di **backend**; kontrak ke aplikasi tidak berubah. Inilah yang membuat kode portabel -- tema yang akan kita lanjutkan di nb02 dengan NVIDIA NIM.

## 5. Intip ke depan -- NIM sebagai backend yang sudah dioptimalkan

Membangun engine TensorRT-LLM sendiri itu kerja berat. **NVIDIA NIM** adalah model yang **sudah** dibungkus dengan optimasi ini (TensorRT-LLM di dalamnya) dan disajikan sebagai layanan OpenAI-compatible. Kamu cukup memanggilnya. Kita bahas tuntas di **nb02**, tapi mari intip kode aslinya sekarang supaya temanya nyambung -- dan supaya kamu lihat satu mekanik NVIDIA yang sangat spesifik **secara langsung, bukan disembunyikan di helper**.

Model yang kita pakai: **`nvidia/nemotron-3-nano-30b-a3b`** -- sebuah model *reasoning* (MoE Hybrid Mamba-Attention, 30B parameter / ~3B aktif). Karena ia model reasoning, secara default ia mengeluarkan blok "berpikir" `<think>...</think>` sebelum jawaban. Untuk tugas klasifikasi/ekstraksi yang butuh output bersih, kita **matikan reasoning** lewat parameter berikut pada **setiap** panggilan:

```python
extra_body={"chat_template_kwargs": {"enable_thinking": False}}
```

> Catatan penting: token `/no_think` di dalam prompt **diabaikan** oleh NIM untuk model ini -- satu-satunya cara yang bekerja adalah `extra_body` di atas. Ini mekanik paling NVIDIA-spesifik di seluruh modul, jadi kita tunjukkan inline.

Notebook ini juga membutuhkan **`NVIDIA_API_KEY`** dari Colab Secrets (ikon kunci di kiri). Satu key gratis melayani generator, judge, sekaligus NemoGuard di notebook-notebook berikutnya.

In [ ]:
# --- Kode NIM ASLI (ditampilkan inline, bukan disembunyikan di helper) ---
# Butuh NVIDIA_API_KEY di Colab Secrets. Jika belum punya: https://build.nvidia.com (gratis).
from openai import OpenAI

try:
    from google.colab import userdata
    nvidia_key = userdata.get("NVIDIA_API_KEY")
except Exception:
    import os
    nvidia_key = os.environ.get("NVIDIA_API_KEY")

# NIM = endpoint OpenAI-compatible. Persis pola "one client, many backends":
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=nvidia_key,
)

NIM_MODEL = "nvidia/nemotron-3-nano-30b-a3b"   # model reasoning -> WAJIB matikan thinking

resp = client.chat.completions.create(
    model=NIM_MODEL,
    messages=[
        {"role": "system", "content": "Kamu asisten ringkas. Jawab dalam satu kalimat."},
        {"role": "user", "content": "Apa fungsi Tensor Core pada GPU NVIDIA?"},
    ],
    temperature=0.2,
    max_tokens=120,
    # Mekanik kunci: matikan reasoning <think> pada SETIAP panggilan ke Nemotron.
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(resp.choices[0].message.content.strip())

Itulah backend NIM bekerja: kode klien sama seperti Ollama/vLLM/TensorRT-LLM; bedanya hanya `base_url` dan model. Di balik layar, NVIDIA sudah menerapkan TensorRT-LLM (paged KV-cache, in-flight batching, FP8/INT4) yang baru saja kita pelajari -- tanpa kamu perlu mem-build apa pun.

> Karena pola ini akan dipakai berulang di nb02-nb05, mulai nb02 kita bungkus ke `nvidia_utils.py` sebagai *helper* `nim_chat(...)`. Tapi sekarang kamu sudah melihat **isi sebenarnya** dari helper itu -- termasuk trik `extra_body` -- jadi helper hanyalah DRY, bukan kotak hitam.

## Yang kita pelajari

| Tuas optimasi | Mekanisme | Hasil terukur (T4) |
|---------------|-----------|--------------------|
| **GPU + Tensor Core** | paralelisme ribuan core, hardware matmul | matmul FP16 **5.6x** lebih cepat dari FP32 |
| **Precision FP16** | 16-bit alih-alih 32-bit | memori bobot **2.0x** lebih hemat (6.18 -> 3.10 GB) |
| **Quantization 4-bit** | NF4 + compute fp16 + double-quant | **2.7x** lebih hemat dari FP16 (3.09 -> 1.16 GB) |
| **TensorRT** | compile + fuse kernel (model umum) | engine native per-GPU (konsep) |
| **TensorRT-LLM** | paged KV-cache, in-flight batching, FP8/INT4 | throughput LLM produksi (konsep + `trtllm-serve`) |

**Tiga pelajaran besar:**

1. **GPU cepat untuk LLM karena matmul-nya paralel**, dan **Tensor Core** mempercepatnya lagi -- tapi hanya untuk precision rendah (FP16/FP8). FP16 bukan sekadar hemat memori; ia *menyalakan* Tensor Core.
2. **Precision & quantization adalah tuas memori**: FP32->FP16 (2x) -> 4-bit (2.7x lagi). Ini *inference* quant (deploy), beda niat dengan QLoRA *training* di Modul 04.
3. **TensorRT-LLM adalah tuas compile** untuk produksi; `trtllm-serve` (dan NIM) menyajikannya lewat kontrak **OpenAI yang sama** -- sehingga kode aplikasimu portabel lintas backend.

## Latihan

1. **Skala matmul.** Ubah `n` di `matmul_bench` dari 4096 menjadi 2048, lalu 8192. Apakah *speedup* FP16/FP32 membesar atau mengecil saat matriks lebih besar? Kenapa Tensor Core makin "untung" pada matriks besar? *(Petunjuk: pikirkan rasio komputasi vs lalu-lintas memori.)*

2. **Double-quant on/off.** Ubah `bnb_4bit_use_double_quant` menjadi `False`. Berapa selisih memori 4-bit-nya? Apakah sepadan dengan potensi kehilangan akurasi? *(Petunjuk: double-quant hanya meng-quantize konstanta, jadi penghematannya kecil.)*

3. **Model lebih besar.** Coba muat `Qwen/Qwen2.5-3B-Instruct` dalam **FP16** lalu dalam **4-bit**. Yang mana muat di T4 16 GB? Hitung berapa GB masing-masing dan cocokkan dengan teori (3B params x 2 byte FP16 ~ 6 GB; dibagi ~2.7 untuk 4-bit).

4. **Bandingkan kualitas precision.** Ajukan prompt yang sama ke model FP16 dan 4-bit (sesuaikan kode generate). Apakah jawabannya identik? Di mana 4-bit mulai "tergelincir" -- fakta, angka, atau gaya bahasa?

5. **NIM (butuh API key).** Ganti `enable_thinking` di `extra_body` menjadi `True` pada panggilan NIM. Apa yang muncul di output? Mengapa output mentah ini menyulitkan untuk tugas klasifikasi/ekstraksi yang akan kita lakukan di nb04?

---

## Selanjutnya -> nb02: Serving & Deploy -- Triton, Dynamo, NIM

Kita sudah punya model yang **kecil dan cepat**. Pertanyaan berikutnya: bagaimana menyajikannya ke **banyak user sekaligus**? Di **nb02** kita masuk ke *serving*: **Triton Inference Server** (baseline ujian, dynamic batching), **NVIDIA Dynamo** (flagship GTC 2025), dan **NIM** secara penuh -- termasuk recap RAG-on-NIM. Pola `extra_body` reasoning-off yang baru kamu lihat akan menjadi fondasi seluruh Track Trustworthy AI (nb03-nb05).

> **Catatan:** Notebook ini dirancang untuk Google Colab dengan GPU T4 (16 GB). Pastikan `Runtime -> Change runtime type -> T4 GPU` aktif.